In [0]:
from pyspark.sql import functions as f
from pyspark.sql.types import StringType

In [0]:
%run ../functions/functions

In [0]:
# Nome do banco de dados onde a tabela será salva
database_name = "dimensao"

# Nome da tabela de destino
table_name = "dm_uf"

# Caminho alvo no formato database.tabela
target_path = f"{database_name}.{table_name}"

# Nome da chave primária da tabela
pk = "PK_UF"

In [0]:
# Define o nome do container de destino onde os dados processados serão armazenados
container_destino = "gold"
 
# Define o nome do container de origem onde os dados brutos estão armazenados
container_origem = "silver"

# Monta o caminho de origem dos dados no Data Lake usando o nome do storage account e o container de origem
caminho_origem = f"abfss://{container_origem}@{STORAGE}.dfs.core.windows.net/balancacomercial"

In [0]:
# Lê a tabela Delta 'UF_MUN' do caminho de origem especificado
df_regiao = spark.read.format("delta").load(f"{caminho_origem}/UF_MUN")

# Seleciona as colunas de interesse: código geográfico do município, nome do município e sigla da UF
df_final = df_regiao.select(
    "CO_MUN_GEO",
    "NO_MUN",
    "SG_UF"
)

# Exibe o DataFrame resultante
df_final.display()

In [0]:
# Salva o DataFrame df_final como uma tabela Hive no caminho especificado, utilizando a chave primária pk
save_hive_table(df_final, target_path, pk)

In [0]:
%sql

-- Seleciona todos os registros da tabela de dimensão de unidades federativas (UF)
select * from hive_metastore.dimensao.dm_uf